In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.model_selection import KFold, ParameterGrid

def getDataset(data, param):
    df = data.dropna(subset=[param]).copy()
    return df

def nested_cv(df, inner_cv, outer_cv, Classifier, parameter_grid, var, param, sheet_name):
    df = getDataset(df, param)
    df = df.sample(frac=1) 
    X, y = np.array(df[var]), np.array(df[param])

    outer_scores = []
    df_all = {}
    
    for i, (training_samples, test_samples) in enumerate(outer_cv.split(X, y), start=1):
        best_params = {}
        best_score = -np.inf

        for parameters in parameter_grid:
            cv_scores = []
            for inner_train, inner_test in inner_cv.split(X[training_samples], y[training_samples]):
                clf = Classifier(**parameters)
                clf.fit(X[inner_train], y[inner_train])
                score = clf.score(X[inner_test], y[inner_test])
                cv_scores.append(score)
                
            mean_score = np.mean(cv_scores)
            if mean_score > best_score:
                best_score = mean_score
                best_params = parameters
                
        clf = Classifier(**best_params)
        clf.fit(X[training_samples], y[training_samples])
        outer_scores.append(clf.score(X[test_samples], y[test_samples]))
        
        ipdict = {n: s for n, s in zip(var, clf.feature_importances_)}
        df_all[i] = pd.Series(ipdict)
        
    results = pd.concat(df_all, axis=1)
    
    mean_series = results.mean(axis=1).rename("mean")
    sd_series = results.std(axis=1, ddof=1).rename("sd")
    
    table = pd.concat([results, mean_series, sd_series], axis=1)
    table = table.sort_values('mean', ascending=False)
    
    scores_dict = {i+1: outer_scores[i] for i in range(len(outer_scores))}
    scores_dict['mean'] = np.mean(outer_scores)
    scores_dict['sd'] = np.std(outer_scores, ddof=1)
    score_df = pd.DataFrame(scores_dict, index=['Model_Score_R2'])
    
    final_table = pd.concat([score_df, table], axis=0)

    out_filename = f"{sheet_name}_{param}.csv"
    final_table.to_csv(Path.cwd() / out_filename)

In [ ]:
df = pd.read_excel(Path.cwd().parent / "data/Data.xlsx", sheet_name="na", 
                   usecols=range(3, 44))
X_cols = df.columns.to_list()

In [ ]:
target_params = ['MaxOD_ave', 'LagTime_ave', 'MaxRate_ave']
use_cols = X_cols + target_params

In [ ]:
param_grid = {
    "learning_rate": [i/100 for i in range(1, 51, 1)],  
    "max_depth": [3, 4, 5],
    "n_estimators": [300]
}

sheet_names = ["na", "multi", "kan", "nal", "ery", "rif"]

for sheet in sheet_names:
    print(f"Processing: sheet '{sheet}'")
    
    try:
        data = pd.read_excel(Path.cwd().parent / "data/Data.xlsx", 
                             sheet_name=sheet, usecols=lambda x: x in use_cols)
        
        available_targets = [col for col in target_params if col in data.columns]
        X = data[X_cols]
        
        min_nonzero = X[X > 0].min().min()
        X_log = np.log10(X + min_nonzero)
        
        y = data[available_targets]
        data_log = pd.concat([X_log, y], axis=1)
        
        for param in available_targets:
            print(f"  -> Processing {param} ...")
            nested_cv(
                df=data_log,
                inner_cv=KFold(5, shuffle=True),
                outer_cv=KFold(5, shuffle=True),
                Classifier=GradientBoostingRegressor,
                parameter_grid=ParameterGrid(param_grid),
                var=X_cols,       
                param=param,
                sheet_name=sheet,
            )
    except Exception as e:
        print(f"Error (sheet '{sheet}'): {e}")